# BIST Algorithmic Trading Challenge
## Python Backtesting & Algorithmic Strategy Implementation

**Ders:** BIST Algorithmic Trading  
**Periyot:** 1 Ocak 2025 - 1 Ekim 2026  
**Baslangic Sermayesi:** Hisse basina 100,000 TL  
**Kapsanan Hisseler:** AKBNK, ASELS, TUPRS, TCELL, FROTO, EREGL  

---

### Proje Kurallari ve Benchmark Hedefleri

| Hisse | Baslangic Sermaye | Benchmark Net Kar | Benchmark Final Sermaye |
|:---|:---:|:---:|:---:|
| **AKBNK** | 100,000 TL | +84,000 TL | 184,000 TL |
| **ASELS** | 100,000 TL | +239,000 TL | 339,000 TL |
| **TUPRS** | 100,000 TL | +72,000 TL | 172,000 TL |
| **TCELL** | 100,000 TL | -18,000 TL | 82,000 TL |
| **FROTO** | 100,000 TL | +3,000 TL | 103,000 TL |
| **EREGL** | 100,000 TL | +39,000 TL | 139,000 TL |

> **Zorunlu Kurallar:**
> 1. Ayni temel strateji 6 hissenin tamamina uygulanmalidir.
> 2. Toplam portfoy kari tek basina yeterli degildir; **her hisse kendi benchmark'ini bireysel olarak gecmelidir**.
> 3. Her hisse icin en az **3 tamamlanmis islem** uretilmelidir.
> 4. **No Look-Ahead Kurali:** Sadece o ana kadar olan veri kullanilabilir.
> 5. **Risk Yonetimi Zorunlulugu:** Stop-loss, trailing stop, ATR stop veya position sizing mekanizmasi icermelidir.



## 1. Kütüphaneler ve Modüllerin Yüklenmesi


In [1]:
import os
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

# Proje kok dizini
sys.path.insert(0, os.path.abspath("."))

from config import STOCKS, INITIAL_CAPITAL, BENCHMARKS, get_short_name, get_benchmark
from data_loader import load_stock_data, load_all_stocks
from indicators import add_sma, add_ema, add_rsi, add_atr, add_macd
from strategy_base import StrategyBase
from risk_manager import RiskManager
from backtester import Backtester, Trade, BacktestResult
from metrics import calculate_metrics, evaluate_challenge, generate_summary_table, print_metrics_summary
from visualizer import plot_stock_dashboard, plot_benchmark_comparison, plot_all_equity_curves, plot_all_dashboards
from runner import run_pipeline

print("[OK] Tum moduller basariyla import edildi.")
print(f"Hisseler: {', '.join([get_short_name(s) for s in STOCKS])}")


[OK] Tum moduller basariyla import edildi.
Hisseler: AKBNK, ASELS, TUPRS, TCELL, FROTO, EREGL


## 2. BIST Hisse Verilerinin Yüklenmesi ve İncelenmesi
Her hisse için 2025-01-01 ile 2026-10-01 tarihleri arasındaki günlük OHLCV verileri yüklenir.


In [2]:
# Verileri yukle ve ozet tablosunu olustur
data_summary = []
all_data = {}

for ticker in STOCKS:
    df = load_stock_data(ticker, use_cache=True)
    all_data[ticker] = df
    short = get_short_name(ticker)
    data_summary.append({
        "Hisse": short,
        "Ticker": ticker,
        "Gun Sayisi": len(df),
        "Baslangic": df.index[0].strftime("%Y-%m-%d"),
        "Bitis": df.index[-1].strftime("%Y-%m-%d"),
        "Ilk Fiyat (TL)": round(df["Close"].iloc[0], 2),
        "Son Fiyat (TL)": round(df["Close"].iloc[-1], 2),
        "Getiri (%)": round(((df["Close"].iloc[-1] / df["Close"].iloc[0]) - 1) * 100, 2),
    })

df_data_summary = pd.DataFrame(data_summary)
print(df_data_summary.to_string(index=False))


  [OK] AKBNK: 440 gun | 2025-01-02 -> 2026-09-30 | Fiyat: 62.74 -> 67.00 TL
  [OK] ASELS: 440 gun | 2025-01-02 -> 2026-09-30 | Fiyat: 73.41 -> 336.50 TL
  [OK] TUPRS: 440 gun | 2025-01-02 -> 2026-09-30 | Fiyat: 125.32 -> 376.25 TL
  [OK] TCELL: 440 gun | 2025-01-02 -> 2026-09-30 | Fiyat: 92.35 -> 96.10 TL
  [OK] FROTO: 440 gun | 2025-01-02 -> 2026-09-30 | Fiyat: 83.93 -> 73.55 TL
  [OK] EREGL: 440 gun | 2025-01-02 -> 2026-09-30 | Fiyat: 23.95 -> 35.94 TL
Hisse   Ticker  Gun Sayisi  Baslangic      Bitis  Ilk Fiyat (TL)  Son Fiyat (TL)  Getiri (%)
AKBNK AKBNK.IS         440 2025-01-02 2026-09-30           62.74           67.00        6.78
ASELS ASELS.IS         440 2025-01-02 2026-09-30           73.41          336.50      358.41
TUPRS TUPRS.IS         440 2025-01-02 2026-09-30          125.32          376.25      200.23
TCELL TCELL.IS         440 2025-01-02 2026-09-30           92.35           96.10        4.07
FROTO FROTO.IS         440 2025-01-02 2026-09-30           83.93           7

## 3. Strateji Mimarisi (StrategyBase) ve Sinyal Üretimi
Strateji nesnemiz `StrategyBase` sınıfından türetilmiştir. Sadece o ana kadar olan verileri kullanarak `Signal` kolonu üretir:
- `1`: BUY (Alış)
- `-1`: SELL (Satış)
- `0`: HOLD (Bekle)


In [3]:
from strategies.sma_crossover import SmaCrossoverStrategy

# Strateji ornegi: SMA Crossover (Fast=10, Slow=50)
strategy = SmaCrossoverStrategy(fast_period=10, slow_period=50)
print(f"Secilen Strateji: {strategy}")

# Ornek olarak AKBNK uzerinde uretilen sinyaller
df_sample = strategy.run(all_data["AKBNK.IS"])
print("\nSinyal Dagilimi (AKBNK):")
print(df_sample["Signal"].value_counts())


Secilen Strateji: SMA_Crossover(fast_period=10, slow_period=50)

Sinyal Dagilimi (AKBNK):
Signal
 0    430
-1      5
 1      5
Name: count, dtype: int64


## 4. Risk Yönetimi (RiskManager)
Şartnameye uygun olarak stratejiden bağımsız risk yönetimi mekanizmaları tanımlanır:
- **Stop-Loss:** %5 sabit kayıp koruması
- **Trailing Stop:** %3 zirveden geri çekilmede kârı kilitleme
- **Position Sizing:** Sermaye korumalı lot hesabı


In [4]:
# Risk Yoneticisi Tanimlama
rm = RiskManager(
    stop_loss_pct=0.05,       # %5 Stop-Loss
    trailing_stop_pct=0.03,   # %3 Trailing Stop
    max_holding_days=None,    # Istege bagli maksimum bar suresi
    position_size_pct=1.0,    # Tum nakit ile pozisyon boyutlama
)

print(f"Aktif Risk Mekanizmalari: {rm.get_active_mechanisms()}")


Aktif Risk Mekanizmalari: ['Stop-Loss: 5.0%', 'Trailing Stop: 3.0%', 'Position Size: 100%']


## 5. End-to-End Backtest Simülasyonu
Tüm 6 hisse üzerinde `run_pipeline` çalıştırılarak emirler (No Look-Ahead garantili `next_open` modunda) yürütülür.


In [5]:
# Pipeline calistir (Tum hisseler + risk yonetimi + metrikler + grafikler)
results, challenge_eval, df_summary = run_pipeline(
    strategy=strategy,
    risk_manager=rm,
    execution_mode="next_open",
    save_charts=True,
    save_trades=True,
    save_metrics=True,
    print_summary=True,
)


BIST TRADING CHALLENGE RUNNER: SMA_Crossover
  Hisseler         : AKBNK, ASELS, TUPRS, TCELL, FROTO, EREGL
  Baslangic Sermaye: 100,000 TL (Hisse basina)
  Emir Modu        : next_open
  Risk Yonetimi    : Stop-Loss: 5.0%, Trailing Stop: 3.0%, Position Size: 100%

>> [AKBNK] Analiz ediliyor...
  [OK] AKBNK: 440 gun | 2025-01-02 -> 2026-09-30 | Fiyat: 62.74 -> 67.00 TL
   Trade log kaydedildi: AKBNK_trades.csv (5 islem)

>> [ASELS] Analiz ediliyor...
  [OK] ASELS: 440 gun | 2025-01-02 -> 2026-09-30 | Fiyat: 73.41 -> 336.50 TL
   Trade log kaydedildi: ASELS_trades.csv (2 islem)

>> [TUPRS] Analiz ediliyor...
  [OK] TUPRS: 440 gun | 2025-01-02 -> 2026-09-30 | Fiyat: 125.32 -> 376.25 TL
   Trade log kaydedildi: TUPRS_trades.csv (5 islem)

>> [TCELL] Analiz ediliyor...
  [OK] TCELL: 440 gun | 2025-01-02 -> 2026-09-30 | Fiyat: 92.35 -> 96.10 TL
   Trade log kaydedildi: TCELL_trades.csv (11 islem)

>> [FROTO] Analiz ediliyor...
  [OK] FROTO: 440 gun | 2025-01-02 -> 2026-09-30 | Fiyat: 83.93 -

## 6. Tamamlanmış İşlem Logları (Trade Details)
Şartname gereği her hisse için gerçekleştirilen tüm işlemler (alış/satış tarihleri, fiyatlar, lot sayısı, PnL ve çıkış sebepleri) aşağıda listelenmiştir.


In [6]:
# Her hissenin trade tablosunu goster
for ticker, res in results.items():
    short = get_short_name(ticker)
    print(f"\n{'='*75}\n{short} ISLEM GECMISI ({res.total_trades} Islem)\n{'='*75}")
    df_t = res.trades_df
    if not df_t.empty:
        cols = ["entry_date", "entry_price", "exit_date", "exit_price", "shares", "pnl", "pnl_pct", "exit_reason", "bars_held", "cum_capital"]
        print(df_t[cols].to_string(index=False))
    else:
        print("Hic islem uretilmedi.")



AKBNK ISLEM GECMISI (5 Islem)
entry_date  entry_price  exit_date  exit_price  shares      pnl  pnl_pct   exit_reason  bars_held  cum_capital
2025-06-03      51.4510 2025-06-13     55.2299    1943  7342.38     7.34 trailing_stop          6    107342.38
2025-11-27      62.0125 2025-12-04     64.0526    1730  3529.40     3.29 trailing_stop          5    110871.78
2026-04-21      82.2000 2026-04-22     80.1705    1348 -2735.76    -2.47 trailing_stop          1    108136.02
2026-06-22      82.5000 2026-06-23     80.4130    1310 -2733.97    -2.53 trailing_stop          1    105402.05
2026-08-31      73.4000 2026-09-02     71.3000    1435 -3013.50    -2.86     stop_loss          2    102388.55

ASELS ISLEM GECMISI (2 Islem)
entry_date  entry_price  exit_date  exit_price  shares      pnl  pnl_pct   exit_reason  bars_held  cum_capital
2025-12-16        212.2 2025-12-17     206.222     471 -2815.63    -2.82     stop_loss          1     97184.37
2026-08-19        380.5 2026-08-25     397.700    

## 7. Performans Metrikleri ve Benchmark Karşılaştırması
Tüm 6 hissenin şartnamedeki metrikleri ve bireysel benchmark hedeflerine göre durumu:


In [7]:
# Ozet karsilastirma tablosu
print(df_summary.to_string(index=False))

print(f"\nGenel Portfoy Net Kari : {challenge_eval['total_net_profit']:+,.2f} TL ({challenge_eval['total_net_profit_pct']:+.2f}%)")
print(f"Gecen Hisseler          : {', '.join(challenge_eval['passed_stocks']) if challenge_eval['passed_stocks'] else 'Yok'}")
print(f"Kalan Hisseler          : {', '.join(challenge_eval['failed_stocks'])}")
print(f"BIST Challenge Durumu   : {challenge_eval['challenge_status']} [{challenge_eval['passed_count']}/{challenge_eval['total_count']}]")


Hisse  Baslangic (TL)  Final (TL)  Net Kar (TL)  Net Kar (%)  Benchmark (TL)  Fark (TL) Durum  Trades  Win Rate (%)  Profit Factor  Max DD (%)  Sharpe  Avg Trade (TL)
AKBNK        100000.0   102388.55       2388.55         2.39        184000.0  -81611.45  FAIL       5         40.00           1.28        9.38    0.21          477.71
ASELS        100000.0   101570.37       1570.37         1.57        339000.0 -237429.63  FAIL       2         50.00           1.56        2.82    0.21          785.18
TUPRS        100000.0   106241.69       6241.69         6.24        172000.0  -65758.31  FAIL       5         40.00           1.65        9.77    0.42         1248.34
TCELL        100000.0    93759.19      -6240.81        -6.24         82000.0   11759.19  PASS      11         27.27           0.54        9.75   -0.52         -567.35
FROTO        100000.0   106303.01       6303.01         6.30        103000.0    3303.01  PASS       3         66.67           2.92        5.34    0.66         2101.0

## 8. Görselleştirmeler ve Grafikler
Her hissenin 3-in-1 paneli (Fiyat + Emirler, Equity Curve + Benchmark, PnL Barları) ve portföy karşılaştırma grafikleri `results/charts/` dizininde üretilmiştir.


In [8]:
# Grafik dosyalarini listele
charts = list(Path("results/charts").glob("*.png"))
print(f"Uretilen Grafik Sayisi: {len(charts)}")
for c in sorted(charts):
    print(f"  - {c.name} ({c.stat().st_size / 1024:.1f} KB)")


Uretilen Grafik Sayisi: 8
  - AKBNK_dashboard.png (327.1 KB)
  - all_equity_curves.png (149.6 KB)
  - ASELS_dashboard.png (267.2 KB)
  - benchmark_comparison.png (130.5 KB)
  - EREGL_dashboard.png (292.5 KB)
  - FROTO_dashboard.png (320.4 KB)
  - TCELL_dashboard.png (367.8 KB)
  - TUPRS_dashboard.png (282.2 KB)


## 9. Yeni Strateji Ekleme ve Test Etme Rehberi
Proje mimarisi tam modüler tasarlanmıştır. Yeni bir algoritma denemek için sadece şu adımları izleyin:

```python
# 1. strategies/altina yeni_strateji.py olustur:
from strategy_base import StrategyBase
from indicators import add_rsi, add_bollinger_bands

class MyCustomStrategy(StrategyBase):
    def generate_signals(self, df):
        df["Signal"] = 0
        # ... sinyal kurallarini yaz ...
        return df

# 2. runner.py veya bu notebook'ta calistir:
my_strat = MyCustomStrategy()
results, eval_res, df_sum = run_pipeline(my_strat, rm)
```

Hiçbir zaman backtester, metrics, visualizer veya data loader modüllerini değiştirmeniz gerekmez!

